# UMAP：比 t-SNE 更快更好的非线性降维
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：03_无监督学习/降维 | 源文件：UMAP.py | 核心功能：非线性降维、n_neighbors/min_dist 调参、支持 transform 新数据

## 概述

UMAP（Uniform Manifold Approximation and Projection）是 2018 年提出的非线性降维算法，迅速成为 t-SNE 的主流替代品。它的三大优势：**更快**（适合大数据集）、**更好地保持全局结构**、**支持 transform 新数据**。

理论基础来自黎曼几何和代数拓扑，但实际使用起来和 t-SNE 一样简单。

脚本在 Iris 数据集上对比了 UMAP、t-SNE 和 PCA 的效果，并展示了 n_neighbors、min_dist 和 metric 参数的影响。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import load_iris, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
# --- 导入库 ---
from sklearn.neighbors import KNeighborsClassifier

# 尝试导入 umap,不可用则跳过
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
# --- 输出结果 ---
    print("[SKIP] UMAP 未安装，跳过本示例")
    import sys; sys.exit(0)
    HAS_UMAP = False
    print("UMAP 未安装,请运行: pip install umap-learn")
    print("以下演示仅展示 UMAP 的使用方式和参数说明\n")

## 数学原理

### 1. UMAP 的理论基础：黎曼几何与代数拓扑

**代码对应**：`umap.UMAP(n_neighbors=15, min_dist=0.1)` 训练 UMAP。

UMAP（Uniform Manifold Approximation and Projection）基于三个假设：
1. 数据均匀分布在黎曼流形上
2. 黎曼度量是局部恒定的（或近似恒定）
3. 流形是局部连通的

### 2. 高维模糊拓扑构建

**步骤 1**：对每个样本 $\mathbf{x}_i$，找到 $k$ 个最近邻（`n_neighbors` 参数）。

**步骤 2**：构建加权图，边权重为：

$$w_{ij} = \exp\left(-\frac{d(\mathbf{x}_i, \mathbf{x}_j) - \rho_i}{\sigma_i}\right)$$

其中 $\rho_i$ 为 $\mathbf{x}_i$ 到其最近邻的距离（局部归一化），$\sigma_i$ 由 $k$ 近邻的约束确定。

**步骤 3**：对称化（模糊联合）：

$$p_{ij} = w_{ij} + w_{ji} - w_{ij} \cdot w_{ji}$$

这构建了高维数据的**模糊单纯复形**（fuzzy simplicial complex）。

### 3. 低维嵌入优化

在低维空间中，使用**逻辑曲线**（logistic curve）作为核函数：

$$q_{ij} = \frac{1}{1 + a\|\mathbf{y}_i - \mathbf{y}_j\|^{2b}}$$

其中 $a$ 和 $b` 由 `min_dist` 参数确定。$

优化目标为交叉熵：

$$\text{CE} = \sum_{i \neq j}\left[p_{ij}\log\frac{p_{ij}}{q_{ij}} + (1-p_{ij})\log\frac{1-p_{ij}}{1-q_{ij}}\right]$$

与 t-SNE 只有 KL 散度（$p\log(p/q)$）不同，UMAP 的交叉熵还包含 $(1-p)\log((1-p)/(1-q))$ 项——这使得 UMAP 能更好地保持**全局结构**。

### 4. UMAP vs t-SNE

| 特性 | t-SNE | UMAP |
|------|-------|------|
| 理论基础 | 概率分布匹配 | 黎曼几何 + 代数拓扑 |
| 损失函数 | KL 散度 | 交叉熵 |
| 全局结构 | 差 | 较好 |
| 计算速度 | 慢 $O(n^2)$ | 快 $O(n^{1.14})$ |
| 可复现性 | 不稳定 | 较稳定 |
| 新数据投影 | 不支持 | 支持 `transform` |

### 5. 关键参数

- `n_neighbors`：局部邻域大小。大值保持全局结构，小值保持局部细节
- `min_dist`：低维空间中点的最小距离。小值使嵌入更紧凑，大值使嵌入更松散
- `n_components`：目标维度（通常 2 或 3）
- `metric`：距离度量（euclidean、cosine、manhattan 等）

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X = StandardScaler().fit_transform(X)

### 2. 基础 UMAP

运行 2. 基础 UMAP 的代码，观察算法在该环节的行为。

In [ ]:
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = reducer.fit_transform(X)

print("=== UMAP 降维（Iris: 4D → 2D）===")
print(f"降维后形状: {X_umap.shape}")
for c in range(3):
    mask = y == c
    print(f"  类别{c} ({iris.target_names[c]}): "
# --- 继续 ---
          f"中心=({X_umap[mask, 0].mean():.2f}, {X_umap[mask, 1].mean():.2f})")

### 3. n_neighbors 参数

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== n_neighbors 参数影响 ===")
for nn in [5, 10, 15, 30, 50, 100]:
    r = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=0.1, random_state=42)
    X_n = r.fit_transform(X)
    knn = KNeighborsClassifier(n_neighbors=5)
# --- 继续 ---
    acc = cross_val_score(knn, X_n, y, cv=5).mean()
    print(f"  n_neighbors={nn:>3}: 1-NN CV准确率={acc:.4f}")

### 4. min_dist 参数

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== min_dist 参数影响 ===")
for md in [0.0, 0.05, 0.1, 0.25, 0.5, 1.0]:
    r = umap.UMAP(n_components=2, n_neighbors=15, min_dist=md, random_state=42)
    X_m = r.fit_transform(X)
    print(f"  min_dist={md}: 点间最小距离={md}")

### 5. 不同维度

运行 5. 不同维度 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 不同 n_components ===")
for dim in [2, 3, 5, 10]:
    r = umap.UMAP(n_components=dim, random_state=42)
    X_d = r.fit_transform(X)
    knn = KNeighborsClassifier(n_neighbors=5)
# --- 继续 ---
    acc = cross_val_score(knn, X_d, y, cv=5).mean()
    print(f"  n_components={dim:>2}: 降维后形状={X_d.shape}, 1-NN CV准确率={acc:.4f}")

### 6. metric 参数

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== 距离度量对比 ===")
for metric in ["euclidean", "cosine", "manhattan", "chebyshev"]:
    r = umap.UMAP(n_components=2, metric=metric, random_state=42)
    X_m = r.fit_transform(X)
    knn = KNeighborsClassifier(n_neighbors=5)
# --- 继续 ---
    acc = cross_val_score(knn, X_m, y, cv=5).mean()
    print(f"  metric={metric:>12}: 1-NN CV准确率={acc:.4f}")

### 7. UMAP vs t-SNE vs PCA

运行 7. UMAP vs t-SNE vs PCA 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.manifold import TSNE
print("\n=== UMAP vs t-SNE vs PCA ===")
pca_2d = PCA(n_components=2).fit_transform(X)
tsne_2d = TSNE(n_components=2, init="pca", random_state=42).fit_transform(X)
umap_2d = umap.UMAP(n_components=2, random_state=42).fit_transform(X)

knn = KNeighborsClassifier(n_neighbors=5)
for name, X_d in [("PCA", pca_2d), ("t-SNE", tsne_2d), ("UMAP", umap_2d)]:
    acc = cross_val_score(knn, X_d, y, cv=5).mean()
    print(f"  {name:>6}: 1-NN CV准确率={acc:.4f}")

### 8. 参数说明（即使未安装也展示）

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== UMAP 参数说明 ===")
print("- n_components: 目标维度（通常 2 或 3）")
print("- n_neighbors: 邻域大小,越大越关注全局结构（默认 15）")
print("- min_dist: 簇内点的最小距离,越小簇越紧凑（默认 0.1）")
print("- metric: 距离度量（euclidean, cosine, manhattan 等）")
# --- 输出结果 ---
print("- random_state: 随机种子,保证可复现")
print()
print("=== UMAP 要点 ===")
print("- 比 t-SNE 快得多,适合大数据集")
print("- 比 t-SNE 更好地保持全局结构")
# --- 输出结果 ---
print("- 可以用于特征工程（不只是可视化）,因为支持 transform")
print("- 支持监督和半监督降维（transform_supervised）")
print("- 学术上比 t-SNE 更有理论支撑（基于黎曼几何和代数拓扑）")

## 关键代码解释

### 两个核心参数

```python
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
```

- **n_neighbors**：邻域大小。越大越关注全局结构（大尺度模式），越小越关注局部细节。默认 15 通常效果不错。
- **min_dist**：嵌入空间中点之间的最小距离。越小簇越紧凑（容易看出簇的边界），越大簇越松散。

### 支持多种距离度量

```python
for metric in ["euclidean", "cosine", "manhattan", "chebyshev"]:
```

UMAP 支持丰富的距离函数。cosine 适合文本向量，euclidean 适合数值数据。

## 使用示例

```python
import umap
X_2d = umap.UMAP(n_components=2, random_state=42).fit_transform(X)
```

## 注意事项

1. **需要安装 umap-learn**：pip install umap-learn
2. **结果有随机性**：设置 random_state 保证可复现
3. **n_neighbors 调参**：默认 15，大数据集可以增大到 50-100
4. **可以 transform 新数据**：
educer.transform(X_new)
5. **理论争议**：部分学者质疑其数学基础，但实际效果广泛认可

## 总结与延伸

以上代码展示了 **UMAP** 的完整流程。

**解读要点**：
- 观察降维后数据的 **可分离性**——同类样本是否聚集
- 对比不同降维方法的可视化效果
- 关注 **方差解释比**（PCA）或 **邻域保持度**（UMAP）

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **监督 UMAP**：利用标签信息指导降维，同类更紧凑
- **半监督 UMAP**：只用部分标签
- **UMAP 作为特征工程**：降维后接分类器，效果有时比原始特征更好
- **Parametric UMAP**：用神经网络学习 UMAP 映射，支持快速 transform
